In [27]:
import json
import xml.etree.ElementTree as ET
import hashlib
import configparser

from collections import OrderedDict # added for ordered dict

In [28]:
config = configparser.ConfigParser()
config.read("config.ini")

INPUT_FILENAME = config.get('config','trace_xes')
OUTPUT_FILENAME = config.get('config','trace_json')
# optional user-provided classifier
# keys separated by whitespace, e.g. "concept:name lifecycle:transition"
CLASSIFIER = None

In [29]:
# Read the file content and compute its hash
with open(INPUT_FILENAME) as f:
    log_string = "".join(f.readlines())
log_hash = hash(log_string)
del log_string

In [30]:
events = []
objects = []
event_object = []
object_object = []

In [31]:
existing_objects = {}
#existing_events = {}

In [32]:
tree = ET.parse(INPUT_FILENAME)
root = tree.getroot()
#TODO: check  in classifier if there is only concept name or lifecycle transition as well

In [33]:
#root.findall('.//{http://www.xes-standard.org/}classifier')[0].get('keys')

In [34]:
# get the first classifier
classifiers = root.findall('.//{http://www.xes-standard.org/}classifier')
if CLASSIFIER:
    # give the user the opportunity to specify classifier
    classifier = CLASSIFIER
elif classifiers:
    # or use the first classifier in the log
    classifier = classifiers[0].get('keys')   
else:
    # or fall back to concept:name and lifecycle:transition if no other information is provided
    classifier = "concept:name lifecycle:transition"
#classifier = CLASSIFIER if CLASSIFIER is not None else classifiers[0].get('keys')
classifier = tuple(classifier.split(" "))

In [35]:
event_id_counter = 1
object_id_counter = 1
case_id_counter = 1

In [36]:
for case in root.findall('.//{http://www.xes-standard.org/}trace'):
    case_attributes = [child for child in case.iter() if child.tag != '{http://www.xes-standard.org/}event']
    xes_case_id = None
    # check if case ID is present in the XES file
    for attr in case_attributes:
        if attr.get('key') == 'concept:name':
            xes_case_id = attr.get('value')
            break
    # create a case object REGARDLESS of whether the case has an ID
        # case ID depending on position, XES case ID (if present) and log hash
    case_id = f"case_{case_id_counter}_{xes_case_id}_{log_hash}" if xes_case_id else f"case_{case_id_counter}_{log_hash}"
    case_id_counter += 1
    # append "o_" to the beginning of hashed case ID to prevent hashes starting from a number
    # "o_" stands for object
    # TODO: should we replace it with "c_" for case?
    case_id_hashed = f"o_{hashlib.sha1(case_id.encode()).hexdigest()}"
    objects.append({"id": case_id_hashed, "instance_of_O": "case", "attributes": [{"value_of_O": "concept:name", "object_attribute_value": xes_case_id}]})
    
    for attr in case_attributes:
        if (attr.get('key') is not None) and (attr.get('key') != 'concept:name'):
            # add object
            object_type = attr.get('key')
            object_value = attr.get('value')
            object_key = f"{object_type}_{object_value}_{log_hash}"
             #generate a SHA-1 hash of the object key
            if object_key not in existing_objects:
                # append "o_" to the beginning of hashed object ID to prevent hashes starting from a number
                object_hash = f"o_{hashlib.sha1(object_key.encode()).hexdigest()}"
                existing_objects[object_key] = {"id": object_hash, "count": 1}
                objects.append({"id": object_hash, "instance_of_O": object_type, "attributes": [{"value_of_O": object_type, "object_attribute_value": object_value}]})
            else:
                existing_objects[object_key]["count"] += 1

            object_id_hashed = existing_objects[object_key]["id"]
            object_object.append({"from": case_id_hashed, "to": object_id_hashed, "relation_type": "case_object"})
       
    for event in [child for child in case.iter() if child.tag == '{http://www.xes-standard.org/}event']:
        # identify event by its position
        event_id = f"event_{event_id_counter}_{log_hash}"
        event_id_counter += 1
        # append "e_" to the beginning of hashed event ID to prevent hashes starting from a number
        event_id_hashed = f"e_{hashlib.sha1(event_id.encode()).hexdigest()}"
        event_time_element = event.find('.//{http://www.xes-standard.org/}date')
        event_time_iso8601 = None

        if event_time_element is not None:
            event_time = event_time_element.attrib.get('value')
            # event time to ISO 8601 format 
            if event_time:
                event_time_iso8601 = event_time.replace("T", " ").replace("Z", "")

        # take all attributes from classifier
        event_classifier = OrderedDict.fromkeys(classifier)

        event_attributes = [attr for attr in event.iter()]
        for attr in event_attributes:
            # fill in event_classifier with values present in the event
            if attr.get('key') in event_classifier:
                event_classifier[attr.get('key')] = attr.get('value')
            # if attr.get('key') == 'concept:name':
            #     concept_name = attr.get('value')
            # elif attr.get('key') == 'lifecycle:transition':
            #     lifecycle_transition = attr.get('value')
        
        # order of attribute values is preserved for all events
        # skip None values, i.e. attributes not present in the event
        event_type = " ".join([v for v in event_classifier.values() if v is not None])
        
        # add the event
        # only add the attributes that are present
        events.append({"id": event_id_hashed, "observed_at": event_time_iso8601, "instance_of": event_type, 
                       "attributes":[{"value_of": k, "event_attribute_value": v} for k,v in event_classifier.items() if v is not None]})
            
        # add event_object relation to the case
        event_object.append({"eventID": event_id_hashed, "objectID": case_id_hashed, "qualifier": "case_event"})

            
        # loop to Iterate through attributes of the event
        for attr in event_attributes:
            # only attributes that are not in the classifier and not timestamp are converted to objects
            # timestamp is also not an attribute
            #if attr.get('key') != 'concept:name' and attr.get('key') != 'lifecycle:transition' and attr.get('key') != 'time:timestamp':
            if (attr.get('key') is not None) and (attr.get('key') not in event_classifier) and (attr.get('key') != 'time:timestamp'):
                # object already exists in objects list? if yes then take its id as an object_id
                object_type = attr.get('key')
                object_value = attr.get('value')
                object_key = f"{object_type}_{object_value}_{log_hash}"
                 #generate a SHA-1 hash of the object key
                if object_key not in existing_objects:
                    object_hash = f"o_{hashlib.sha1(object_key.encode()).hexdigest()}"
                    existing_objects[object_key] = {"id": object_hash, "count": 1}
                    objects.append({"id": object_hash, "instance_of_O": object_type, "attributes": [{"value_of_O": object_type, "object_attribute_value": object_value}]})
                else:
                    existing_objects[object_key]["count"] += 1

                object_id_hashed = existing_objects[object_key]["id"]

                event_object.append({"eventID": event_id_hashed, "objectID": object_id_hashed, "qualifier": "event_object"})

In [37]:
if len(object_object) < 1:
    # if this part is empty, add an empty entry for demonstrative purposes
    object_object.append({"from": "", "to": "", "object_relation_type": ""})

In [38]:
output_data = {
    "events": events,
    "objects": objects,
    "event_object": event_object,
    "object_object": object_object
}

In [39]:
with open(OUTPUT_FILENAME, 'w') as json_file:
    json.dump(output_data, json_file, indent=4)

print(f"Output saved to {OUTPUT_FILENAME}")

Output saved to output/2013_small.json
